# 01 — LangChain Overview

**Goal**: Understand LangChain's core abstractions and build your first chain.

LangChain is a framework that makes it easy to build LLM-powered applications.
It wraps the raw API calls you wrote in Module 2 into reusable components.

In [ ]:
# !pip install langchain langchain-community

from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.schema import BaseOutputParser

print("Imports OK")

## 1. LangChain's Core Abstractions

```
┌──────────────────────────────────────────────────┐
│                   LangChain                      │
│                                                  │
│  Models        Prompts          Chains           │
│  ┌──────┐    ┌────────┐    ┌──────────────┐    │
│  │LLM   │    │Template│    │Prompt→LLM    │    │
│  │Chat  │    │Example │    │  →Output Parse│    │
│  │Embed │    │Selector│    │              │    │
│  └──────┘    └────────┘    └──────────────┘    │
│                                                  │
│  Memory         Agents         Retrievers        │
│  ┌──────┐    ┌──────────┐    ┌────────────┐    │
│  │Buffer│    │Tool      │    │VectorStore │    │
│  │Window│    │Agent     │    │Retriever   │    │
│  │Summ  │    │Executor  │    │            │    │
│  └──────┘    └──────────┘    └────────────┘    │
└──────────────────────────────────────────────────┘
"""

## 2. Creating a LangChain LLM Wrapper

Instead of calling `requests.post()` manually, LangChain wraps Ollama.

In [ ]:
llm = Ollama(model="llama3.2")
result = llm.invoke("Explain what LangChain is in one sentence.")
print(result)

## 3. Prompt Templates

Instead of f-strings, LangChain uses `PromptTemplate` objects.

In [ ]:
template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="""Explain {topic} to someone who is {audience}. Use simple language."""
)

# Format the template (without calling LLM)
formatted = template.format(topic="RAG", audience="a complete beginner")
print("Formatted prompt:")
print(formatted)

## 4. Building a Chain

A chain connects: Prompt → LLM → Output Parser

In [ ]:
# Simple chain
chain = LLMChain(llm=llm, prompt=template)

result = chain.run(topic="vector databases", audience="a high school student")
print(result)

## 5. Custom Output Parser

Ensure the model outputs exactly what you want.

In [ ]:
class CommaSeparatedParser(BaseOutputParser):
    """Parse comma-separated list from LLM output."""
    def parse(self, text: str):
        return [item.strip() for item in text.split(",")]

parser = CommaSeparatedParser()

# Test parsing
test = parser.parse("Python, JavaScript, Rust, Go")
print(f"Parsed list: {test}")

In [ ]:
# Chain with parser
chain_with_parser = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["topic"],
        template="List 5 key concepts related to {topic}. Return as comma-separated list."
    ),
    output_parser=parser
)

result = chain_with_parser.run(topic="machine learning")
print(f"Parsed result: {result}")
print(f"Type: {type(result)}")

## 6. Chaining Chains Together (Sequential Chain)

Chain 1: Generate a topic → Chain 2: Explain the topic

In [ ]:
from langchain.chains import SimpleSequentialChain

# Chain 1: Generate a fact
chain1 = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["subject"],
        template="Give me one interesting fact about {subject}."
    )
)

# Chain 2: Explain it
chain2 = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["fact"],
        template="Explain why this fact is important: {fact}"
    )
)

# Combined
overall = SimpleSequentialChain(chains=[chain1, chain2], verbose=True)
result = overall.run("neural networks")
print(f"\nFinal output:\n{result}")

## 7. LangChain Expression Language (LCEL)

Modern LangChain uses the `|` operator (like Unix pipes) to compose chains.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# LCEL style (more modern and flexible)
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a short slogan about {topic}."
)

chain_lcel = prompt | llm | (lambda x: f"Slogan: {x}")

result = chain_lcel.invoke({"topic": "vector databases"})
print(result)

## Key Takeaways

1. **LangChain** standardizes: LLM calls, prompts, chains, parsers
2. **PromptTemplate** → reusable, parameterized prompts
3. **LLMChain** → prompt + LLM + parser in one
4. **Sequential chains** → compose multiple steps
5. **LCEL (`|`)** → modern, pipe-style chain composition

**Next**: Add memory to chains (conversation history) →